# 03 — Feature Engineering Strategy

**Purpose:** This notebook gives every club member a working strategy they can learn from and build on.

We will cover:

1. **What the data looks like** — quick refresher so you know what you are working with
2. **Why feature engineering matters** — short explanation for beginners
3. **Step-by-step feature engineering** — every transformation explained in plain English
4. **Preprocessing pipeline** — winsorization, log transforms, handling missing values
5. **Three models compared** — Ridge, LightGBM, XGBoost
6. **Ensemble** — combining models for a better score
7. **Feature importance** — which features actually matter
8. **Final submission** — generating a submission file

---

### A Note for Beginners

If you are new to machine learning competitions, don't panic. The core idea is simple:

> We have a table of companies. Each row is a company at a specific quarter.
> Each column is a number describing that company (profit margins, debt levels, etc.).
> We want to predict how much the stock price will change over the next year.

**Feature engineering** means creating new columns from existing ones to help the model
find patterns. For example, if we know revenue and net income, we can compute the profit
margin — that ratio might be more useful than either number alone.

Let's get started.

## 1. Setup and Data Loading

We load the competition files and take a quick look at what we have.

In [1]:
# =============================================================================
# Standard imports — these are libraries (pre-written code) we use throughout.
#
#   pandas   → works with tables of data (called DataFrames)
#   numpy    → fast math on arrays of numbers
#   matplotlib / seaborn → plotting
#   sklearn  → machine learning models and utilities
#   lightgbm / xgboost → powerful tree-based models popular on Kaggle
# =============================================================================

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')  # keep output clean

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for batch execution
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import lightgbm as lgb
import xgboost as xgb

from IPython.display import display

# Pretty plots
sns.set_theme(style='whitegrid', font_scale=1.1)
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# Paths — we assume the notebook is in the 'notebooks/' folder
ROOT = Path.cwd().resolve().parent
DATA_DIR = ROOT / 'data' / 'raw'
SUBMISSION_DIR = ROOT / 'submissions'
SUBMISSION_DIR.mkdir(exist_ok=True)

print('Project root:', ROOT)

Project root: C:\Users\joni0\kaggle-competition-stock-return-fundamentals


In [2]:
# =============================================================================
# Load the three competition files:
#   train.csv  → rows we can learn from (has the answer: 'return_pct')
#   test.csv   → rows we must predict (no answer column)
#   sample_submission.csv → template showing the expected output format
#
# IMPORTANT: train has 'period_start' and 'period_end' columns, but test does NOT.
#            Both have 'start_year' which tells us the observation year.
# =============================================================================

train = pd.read_csv(
    DATA_DIR / 'train.csv',
    parse_dates=['period_start', 'period_end'],  # train has these columns
)
test = pd.read_csv(DATA_DIR / 'test.csv')  # test does NOT have date columns
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f'Training data:   {train.shape[0]:,} rows  ×  {train.shape[1]} columns')
print(f'Test data:       {test.shape[0]:,} rows  ×  {test.shape[1]} columns')
print(f'Submission rows: {sample_submission.shape[0]:,}')
print()
print('Training years:', sorted(train['start_year'].unique()))
print('Test years:    ', sorted(test['start_year'].unique()))

Training data:   23,070 rows  ×  39 columns
Test data:       8,520 rows  ×  36 columns
Submission rows: 8,520

Training years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Test years:     [np.int64(2024)]


## 2. Quick Data Overview

Before engineering features, we need to understand what we are working with.

### The columns fall into these groups:

| Group | Columns | What they measure |
|-------|---------|-------------------|
| **Identity** | `id`, `ticker` | Which company and row |
| **Time** | `start_year`, `period_start`, `period_end` | When the observation was taken |
| **Valuation** | `pe_ttm`, `price_to_book`, `price_to_sales`, `growth_pe_ratio` | Is the stock cheap or expensive? |
| **Profitability** | `gross_margin`, `operating_margin`, `net_margin`, `roa`, `roe`, `rote` | How good is the company at making profit? |
| **Growth** | `revenue_growth_3y`, `revenue_growth_yoy` | Is the company growing? |
| **Scale** | `revenue_ttm`, `net_income_ttm`, `total_assets`, `shares_outstanding` | How big is the company? |
| **Balance Sheet** | `current_ratio`, `quick_ratio`, `debt_to_equity`, `long_term_debt` | How healthy are its finances? |
| **Dividends** | `dividend_yield`, `dividends_ttm`, `dividends_paid_ttm` | Does it pay shareholders? |
| **Sector** | `sector_code` | What industry is it in? |
| **Target** | `return_pct` | 1-year forward stock return (what we predict) |

### Why this matters:
- **Valuation** features tell us if a stock is cheap → cheap stocks sometimes bounce back
- **Profitability** tells us if the business is good → quality companies tend to hold up
- **Growth** tells us momentum → but high growth is often already priced in
- **Balance sheet** tells us risk → heavily indebted companies can crash

In [3]:
# =============================================================================
# Let's look at the target variable: return_pct
#
# This is what we are trying to predict: the stock's return over the next year,
# expressed as a percentage. For example:
#   return_pct =  50  means the stock went UP 50%
#   return_pct = -30  means the stock went DOWN 30%
#   return_pct = 0    means it stayed flat
# =============================================================================

target = train['return_pct']

print('=== Target (return_pct) Summary ===')
print(f'  Average return:  {target.mean():.1f}%')
print(f'  Median return:   {target.median():.1f}%')
print(f'  Std deviation:   {target.std():.1f}%  ← this is HUGE, meaning lots of variation')
print(f'  Worst return:    {target.min():.1f}%')
print(f'  Best return:     {target.max():.1f}%  ← extreme outlier!')
print()
print('Percentiles:')
for pct in [1, 5, 25, 50, 75, 95, 99]:
    val = target.quantile(pct / 100)
    print(f'  {pct:3d}th percentile: {val:8.1f}%')

=== Target (return_pct) Summary ===
  Average return:  18.8%
  Median return:   3.5%
  Std deviation:   138.7%  ← this is HUGE, meaning lots of variation
  Worst return:    -99.2%
  Best return:     10571.1%  ← extreme outlier!

Percentiles:
    1th percentile:    -80.2%
    5th percentile:    -56.2%
   25th percentile:    -19.8%
   50th percentile:      3.5%
   75th percentile:     33.8%
   95th percentile:    118.1%
   99th percentile:    299.2%


In [4]:
# =============================================================================
# Target distribution by year
#
# KEY INSIGHT: Different years behave very differently!
# - 2020 had huge returns (COVID recovery rally)
# - 2021 had negative average returns (the market pulled back)
# This is called 'regime dependence' and is why we must validate carefully.
# =============================================================================

year_stats = train.groupby('start_year')['return_pct'].agg(
    ['count', 'mean', 'median', 'std']
).round(2)
year_stats.columns = ['Rows', 'Mean Return %', 'Median Return %', 'Std Dev']
print('Target statistics by year:')
display(year_stats)
print()
print('Notice how 2020 has a much higher mean (73.9%) due to the COVID recovery.')
print('And 2021 has negative returns (-10.5%) — the market environment changed.')
print('This is why a time-based validation split is essential.')

Target statistics by year:


,Rows,Mean Return %,Median Return %,Std Dev
start_year,,,,
2019,5029,4.1000,-7.7000,86.7200
2020,5339,73.8700,42.7900,254.1500
2021,6068,-10.5100,-12.1600,41.1600
2022,6634,12.3600,3.2600,64.7700



Notice how 2020 has a much higher mean (73.9%) due to the COVID recovery.
And 2021 has negative returns (-10.5%) — the market environment changed.
This is why a time-based validation split is essential.


In [5]:
# =============================================================================
# Missing data overview
#
# In finance datasets, missing values are COMMON and MEANINGFUL.
# For example, a company that doesn't report gross_margin might be
# a financial company (banks don't have 'cost of goods sold').
#
# We will use this info later to create 'missingness flags' — binary columns
# that tell the model "this value was missing".
# =============================================================================

missing_pct = (train.isna().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

print(f'{len(missing_pct)} columns have missing values:\n')
for col, pct in missing_pct.items():
    bar = '█' * int(pct / 2)
    print(f'  {col:25s} {pct:5.1f}%  {bar}')

33 columns have missing values:

  dividends_paid_ttm         93.2%  ██████████████████████████████████████████████
  dividend_yield             72.0%  ████████████████████████████████████
  dividends_ttm              70.8%  ███████████████████████████████████
  gross_margin               62.5%  ███████████████████████████████
  inventory                  49.9%  ████████████████████████
  debt_to_equity             37.2%  ██████████████████
  shares_outstanding         35.4%  █████████████████
  growth_pe_ratio            34.7%  █████████████████
  income_before_tax          34.6%  █████████████████
  long_term_debt             33.1%  ████████████████
  current_ratio              32.1%  ████████████████
  quick_ratio                32.1%  ████████████████
  current_liabilities        30.6%  ███████████████
  revenue_growth_3y          30.6%  ███████████████
  current_assets             30.6%  ███████████████
  goodwill                   30.5%  ███████████████
  price_to_book           

## 3. Validation Split

### Why can't we just randomly split the data?

In stock prediction, the future is different from the past. If we randomly mix
2019 and 2022 rows into training and testing, the model learns patterns from
the future and uses them to "predict" the past — that is **leakage**.

Instead, we use a **time-based split**:
- **Train** on 2019, 2020, 2021 data (the past)
- **Validate** on 2022 data (the "future" we pretend to predict)

This mimics the real Kaggle test, which is 2024 data.

```
  2019    2020    2021    2022    2024
  ├──────────────────────┤├──┤    ├──┤
       Training fold   Validation Test (Kaggle)
```

In [1]:
# =============================================================================
# Create the time-based train/validation split
#
# train_fold: rows from 2019-2021 (what we train the model on)
# valid_fold: rows from 2022 (what we evaluate the model on)
#
# RULE: Everything we compute (means, medians, scaling factors) must come
#       from train_fold ONLY. If we peek at valid_fold, that's leakage.
# =============================================================================

train_mask = train['start_year'] < 2022
valid_mask = train['start_year'] == 2022

train_fold = train.loc[train_mask].copy()
valid_fold = train.loc[valid_mask].copy()

print(f'Training fold: {len(train_fold):,} rows (years {train_fold["start_year"].unique()})')
print(f'Validation fold: {len(valid_fold):,} rows (years {valid_fold["start_year"].unique()})')


def rmse(y_true, y_pred):
    """Root Mean Squared Error — the competition metric.
    Lower is better. Penalizes big mistakes more than small ones."""
    return np.sqrt(mean_squared_error(y_true, y_pred))

NameError: name 'train' is not defined

## 4. Feature Engineering — The Core of This Notebook

This is where we create new columns that help the model make better predictions.

### Why bother?

Raw data is often not in the best format for models. Consider two examples:

1. **Revenue** is in absolute dollars. Apple's revenue is $400 billion, a small biotech
   might have $10 million. The raw number is not comparable — but the **profit margin**
   (net income ÷ revenue) is comparable and more meaningful.

2. **P/E ratio of 15** means something completely different for a bank vs a tech company.
   A bank at P/E 15 might be expensive; a tech company at P/E 15 might be cheap.
   Creating a **sector-relative P/E** ("how does this P/E compare to others in the same sector?")
   captures this.

### Our feature engineering strategy has 7 steps:

1. **Winsorize** extreme values (clip outliers)
2. **Log-transform** scale features (revenue, assets, etc.)
3. **Create financial ratios** (earnings yield, payout ratio, etc.)
4. **Create margin spreads** (gross minus operating, operating minus net)
5. **Create sector-relative features** (z-scores within each sector)
6. **Create missingness flags** ("was this value missing?")
7. **Create interaction features** (growth × valuation)

In [7]:
# =============================================================================
# STEP 0: Define which columns belong to which category
#
# This makes the code organized and easy to modify.
# If a column doesn't exist in your data, it will be skipped safely.
# =============================================================================

# Columns we will NOT use as features (identifiers, dates, target)
DROP_COLS = ['id', 'ticker', 'period_start', 'period_end', 'return_pct']

# Columns that represent absolute dollar amounts → need log transform
SCALE_COLS = [
    'revenue_ttm', 'net_income_ttm', 'income_before_tax',
    'total_assets', 'stockholders_equity',
    'current_assets', 'current_liabilities',
    'long_term_debt', 'goodwill', 'inventory',
    'dividends_ttm', 'dividends_paid_ttm',
    'shares_outstanding', 'shares_diluted',
]

# Columns that are ratios/percentages → need winsorization but NOT log transform
RATIO_COLS = [
    'pe_ttm', 'price_to_book', 'price_to_sales', 'growth_pe_ratio',
    'gross_margin', 'operating_margin', 'net_margin',
    'roa', 'roe', 'rote',
    'revenue_growth_3y', 'revenue_growth_yoy',
    'eps_basic', 'eps_diluted',
    'current_ratio', 'quick_ratio', 'debt_to_equity',
    'dividend_yield',
]

# Columns we will compute sector-relative z-scores for
SECTOR_ZSCORE_COLS = [
    'pe_ttm', 'price_to_book', 'price_to_sales',
    'gross_margin', 'operating_margin', 'net_margin',
    'roa', 'roe',
    'revenue_growth_yoy', 'debt_to_equity', 'dividend_yield',
]

# Columns where missingness might carry information
MISSING_FLAG_COLS = [
    'pe_ttm', 'price_to_book', 'gross_margin',
    'dividend_yield', 'debt_to_equity', 'inventory',
    'revenue_growth_3y', 'roe', 'roa',
]

print(f'Scale columns:        {len(SCALE_COLS)}')
print(f'Ratio columns:        {len(RATIO_COLS)}')
print(f'Sector z-score cols:  {len(SECTOR_ZSCORE_COLS)}')
print(f'Missing flag cols:    {len(MISSING_FLAG_COLS)}')

Scale columns:        14
Ratio columns:        18
Sector z-score cols:  11
Missing flag cols:    9


In [8]:
# =============================================================================
# STEP 1: Winsorization (Clipping Extreme Values)
#
# WHAT:  Replace extreme values with the 1st and 99th percentile.
# WHY:   Some columns have absurd outliers (e.g., P/E of 12 billion!).
#        These outliers distort the model and inflate RMSE.
#
# EXAMPLE:
#   Before: [1, 5, 10, 15, 10000]  ← the 10000 warps everything
#   After:  [1, 5, 10, 15, 100]    ← clipped to 99th percentile
#
# IMPORTANT: We compute the clip boundaries from the TRAINING fold only,
#            then apply them to validation and test. This avoids leakage.
# =============================================================================

def compute_clip_bounds(df, columns, lower_pct=0.01, upper_pct=0.99):
    """
    Compute the clipping bounds for each column.
    Returns a dict: {column_name: (lower_bound, upper_bound)}
    """
    bounds = {}
    for col in columns:
        if col in df.columns:
            lower = df[col].quantile(lower_pct)
            upper = df[col].quantile(upper_pct)
            bounds[col] = (lower, upper)
    return bounds


def apply_clip(df, bounds):
    """
    Clip values in the dataframe using pre-computed bounds.
    """
    df = df.copy()
    for col, (lower, upper) in bounds.items():
        if col in df.columns:
            df[col] = df[col].clip(lower=lower, upper=upper)
    return df


# Compute bounds from training fold only
all_numeric_cols = SCALE_COLS + RATIO_COLS
clip_bounds = compute_clip_bounds(train_fold, all_numeric_cols)

# Show a few examples
print('Example clip bounds (computed from training data only):')
for col in ['pe_ttm', 'price_to_book', 'revenue_ttm', 'debt_to_equity']:
    if col in clip_bounds:
        lo, hi = clip_bounds[col]
        print(f'  {col:25s}  [{lo:>15.2f},  {hi:>15.2f}]')

Example clip bounds (computed from training data only):
  pe_ttm                     [       -4391.01,          1207.01]
  price_to_book              [        -131.28,          1041.96]
  revenue_ttm                [     2991320.00,  158423880000.00]
  debt_to_equity             [         -33.03,            61.20]


In [9]:
# =============================================================================
# STEP 2: Log Transform for Scale Features
#
# WHAT:  Apply log(1 + |x|) to absolute-dollar columns, keeping the sign.
# WHY:   Revenue ranges from $0 to $600 billion. On a linear scale, the model
#        sees Apple and a tiny startup as essentially the same (both are tiny
#        compared to the range). Log compresses this:
#        - $10M  → log(10M) ≈ 16.1
#        - $400B → log(400B) ≈ 26.7
#        Now the model can see meaningful differences across the whole range.
#
# We use 'signed log': log1p(|x|) * sign(x), because some values can be negative
# (e.g., negative net income = the company lost money).
# =============================================================================

def signed_log1p(series):
    """
    Apply log1p to the absolute value, then restore the sign.
    This works for both positive and negative values.
    
    log1p(x) = log(1 + x), which is safe for x = 0 (gives 0).
    """
    return np.sign(series) * np.log1p(np.abs(series))


# Quick demonstration
demo_values = pd.Series([0, 1000, 1_000_000, 1_000_000_000, -500_000])
demo_log = signed_log1p(demo_values)
print('Demonstration of signed_log1p:')
print(pd.DataFrame({'original': demo_values, 'log_transformed': demo_log.round(2)}))
print()
print('Notice how the billion-dollar value (1e9) is now just 20.7, not 1,000,000,000.')
print('This makes all values comparable on a similar scale.')

Demonstration of signed_log1p:
     original  log_transformed
0           0           0.0000
1        1000           6.9100
2     1000000          13.8200
3  1000000000          20.7200
4     -500000         -13.1200

Notice how the billion-dollar value (1e9) is now just 20.7, not 1,000,000,000.
This makes all values comparable on a similar scale.


In [10]:
# =============================================================================
# STEP 3: Financial Ratio Features
#
# We create new ratios that financial analysts commonly use.
# These capture relationships between variables that the raw data doesn't show.
#
# Each ratio is explained below.
# =============================================================================

def create_financial_ratios(df):
    """
    Create new financial ratio features from existing columns.
    
    Each ratio is designed to capture a specific financial concept.
    We use .where() and safe division to avoid dividing by zero.
    """
    out = pd.DataFrame(index=df.index)
    
    # --- Earnings Yield ---
    # = 1 / PE ratio.  Higher earnings yield = cheaper stock.
    # Think of it as: "how much profit does the company earn for each dollar
    # of stock price?"  A stock with P/E of 10 has earnings yield of 10%.
    if 'pe_ttm' in df.columns:
        pe = df['pe_ttm'].replace(0, np.nan)
        out['earnings_yield'] = (1.0 / pe).clip(-1, 1)
    
    # --- Profit Margin from Income/Revenue ---
    # Recomputed ratio: net_income / revenue.
    # This is similar to net_margin but computed fresh to double-check.
    if 'net_income_ttm' in df.columns and 'revenue_ttm' in df.columns:
        rev = df['revenue_ttm'].replace(0, np.nan)
        out['computed_net_margin'] = (df['net_income_ttm'] / rev).clip(-5, 5)
    
    # --- Asset Turnover ---
    # = Revenue / Total Assets
    # Measures how efficiently the company uses its assets to generate revenue.
    # Higher = more efficient.
    if 'revenue_ttm' in df.columns and 'total_assets' in df.columns:
        assets = df['total_assets'].replace(0, np.nan)
        out['asset_turnover'] = (df['revenue_ttm'] / assets).clip(-5, 5)
    
    # --- Leverage Ratio ---
    # = Total Assets / Stockholders' Equity
    # Higher leverage = more debt relative to equity = more risk.
    if 'total_assets' in df.columns and 'stockholders_equity' in df.columns:
        equity = df['stockholders_equity'].replace(0, np.nan)
        out['leverage_ratio'] = (df['total_assets'] / equity).clip(-20, 20)
    
    # --- Working Capital Ratio ---
    # = (Current Assets - Current Liabilities) / Total Assets
    # Measures short-term financial health.
    # Positive = company can pay its near-term bills.
    if all(c in df.columns for c in ['current_assets', 'current_liabilities', 'total_assets']):
        assets = df['total_assets'].replace(0, np.nan)
        out['working_capital_ratio'] = (
            (df['current_assets'] - df['current_liabilities']) / assets
        ).clip(-5, 5)
    
    # --- Dividend Payout Ratio ---
    # = Dividends / Net Income
    # How much of its profit does the company return to shareholders?
    # Very high payout ratios can signal either maturity or unsustainability.
    if 'dividends_ttm' in df.columns and 'net_income_ttm' in df.columns:
        income = df['net_income_ttm'].replace(0, np.nan)
        out['payout_ratio'] = (df['dividends_ttm'] / income).clip(-5, 5)
    
    # --- Debt Coverage ---
    # = Long-term Debt / Revenue
    # How many years of revenue would it take to pay off debt?
    if 'long_term_debt' in df.columns and 'revenue_ttm' in df.columns:
        rev = df['revenue_ttm'].replace(0, np.nan)
        out['debt_to_revenue'] = (df['long_term_debt'] / rev).clip(-10, 10)
    
    # --- Book Value Per Share ---
    # = Stockholders' Equity / Shares Outstanding
    # How much net asset value backs each share?
    if 'stockholders_equity' in df.columns and 'shares_outstanding' in df.columns:
        shares = df['shares_outstanding'].replace(0, np.nan)
        bvps = df['stockholders_equity'] / shares
        out['log_book_value_per_share'] = signed_log1p(bvps)
    
    return out


# Test on a small sample
demo_ratios = create_financial_ratios(train_fold.head(5))
print(f'Created {len(demo_ratios.columns)} new ratio features:')
print(list(demo_ratios.columns))
display(demo_ratios)

Created 8 new ratio features:
['earnings_yield', 'computed_net_margin', 'asset_turnover', 'leverage_ratio', 'working_capital_ratio', 'payout_ratio', 'debt_to_revenue', 'log_book_value_per_share']


,earnings_yield,computed_net_margin,asset_turnover,leverage_ratio,working_capital_ratio,payout_ratio,debt_to_revenue,log_book_value_per_share
4,0.0427,0.2345,0.9651,4.8738,0.0447,0.1864,0.6677,1.6382
5,0.0442,0.2500,1.0525,5.1313,0.0202,0.1652,0.6092,1.5856
6,0.0437,0.2588,1.0422,5.5635,0.0267,0.1528,0.5965,1.5771
7,0.0387,0.2658,0.9925,5.2993,0.0146,0.1451,0.5637,1.6868
8,0.0627,0.2135,0.8364,4.0854,0.1487,0.2451,0.6649,2.9516


In [11]:
# =============================================================================
# STEP 4: Margin Spreads
#
# WHAT:  Difference between two related margin metrics.
# WHY:   The spread tells us something the individual margins don't.
#
# Example: If gross_margin is 60% but net_margin is only 5%, there is a
#          55% 'margin loss' between gross and net. This could mean high
#          operating expenses, high interest payments, or high taxes.
#          Companies with large spreads might have more room to improve.
# =============================================================================

def create_margin_spreads(df):
    """Create spread features between margin levels."""
    out = pd.DataFrame(index=df.index)
    
    # Gross - Operating = how much is lost to operating expenses
    if 'gross_margin' in df.columns and 'operating_margin' in df.columns:
        out['gross_minus_operating'] = df['gross_margin'] - df['operating_margin']
    
    # Operating - Net = how much is lost to taxes, interest, etc.
    if 'operating_margin' in df.columns and 'net_margin' in df.columns:
        out['operating_minus_net'] = df['operating_margin'] - df['net_margin']
    
    # ROE - ROA = tells us about the impact of leverage on returns
    # If ROE >> ROA, the company is boosting returns through debt
    if 'roe' in df.columns and 'roa' in df.columns:
        out['roe_minus_roa'] = df['roe'] - df['roa']
    
    return out


demo_spreads = create_margin_spreads(train_fold.head(5))
print(f'Created {len(demo_spreads.columns)} margin spread features:')
print(list(demo_spreads.columns))

Created 3 margin spread features:
['gross_minus_operating', 'operating_minus_net', 'roe_minus_roa']


In [12]:
# =============================================================================
# STEP 5: Sector-Relative Z-Scores
#
# WHAT:  For each stock, compute "how many standard deviations away from
#        the sector average is this value?"
#
# WHY:   A P/E of 30 is normal for a tech company but expensive for a utility.
#        By comparing each stock to its sector peers, we make the features
#        more meaningful.
#
# FORMULA:  z_score = (value - sector_mean) / sector_std
#
# EXAMPLE:
#   Tech sector average P/E = 25, std = 10
#   Stock A P/E = 35  →  z = (35 - 25) / 10 = +1.0  (slightly expensive)
#   Stock B P/E = 15  →  z = (15 - 25) / 10 = -1.0  (cheap for tech)
#
# IMPORTANT: We compute sector means and stds from the TRAINING FOLD only.
# =============================================================================

def compute_sector_stats(df, columns, sector_col='sector_code'):
    """
    Compute mean and std for each column within each sector.
    Returns two DataFrames: means and stds.
    """
    means = df.groupby(sector_col)[columns].mean()
    stds = df.groupby(sector_col)[columns].std()
    # Also compute global mean/std as fallback for missing sectors
    global_means = df[columns].mean()
    global_stds = df[columns].std()
    return means, stds, global_means, global_stds


def apply_sector_zscores(df, columns, sector_means, sector_stds,
                         global_means, global_stds, sector_col='sector_code'):
    """
    Compute z-scores relative to sector peers.
    Falls back to global stats if the sector is unknown.
    """
    out = pd.DataFrame(index=df.index)
    
    for col in columns:
        if col not in df.columns:
            continue
        
        # Map each row's sector to the sector mean/std for this column
        mean_mapped = df[sector_col].map(sector_means[col]).fillna(global_means[col])
        std_mapped = df[sector_col].map(sector_stds[col]).fillna(global_stds[col])
        
        # Avoid division by zero
        std_mapped = std_mapped.replace(0, np.nan)
        
        z = (df[col] - mean_mapped) / std_mapped
        out[f'{col}_sector_z'] = z.clip(-5, 5)  # clip extreme z-scores
    
    return out


# Compute sector stats from training fold
available_zscore_cols = [c for c in SECTOR_ZSCORE_COLS if c in train_fold.columns]
sector_means, sector_stds, global_means, global_stds = compute_sector_stats(
    train_fold, available_zscore_cols
)

print(f'Will create sector-relative z-scores for {len(available_zscore_cols)} features.')
print('Example: sector-average P/E by sector (from training data):')
if 'pe_ttm' in sector_means.columns:
    display(sector_means[['pe_ttm']].round(2))

Will create sector-relative z-scores for 11 features.
Example: sector-average P/E by sector (from training data):


,pe_ttm
sector_code,
0.0000,383.8700
1.0000,-319.0900
2.0000,-917.3300
3.0000,-55.2000
4.0000,-2325.7600
5.0000,77602294.5800
6.0000,-6654.8800
7.0000,-112.5200
8.0000,-1114742.0400


In [13]:
# =============================================================================
# STEP 6: Missingness Flags
#
# WHAT:  Create a binary (0 or 1) column for each selected feature:
#        1 = the value was missing, 0 = it was present.
#
# WHY:   In financial data, missingness is NOT random.
#        - If 'gross_margin' is missing, the company might be a bank
#          (banks don't have traditional "cost of goods sold")
#        - If 'dividend_yield' is missing, the company probably doesn't
#          pay dividends (common for growth stocks)
#        - If 'inventory' is missing, it might be a software/service company
#
#        The MODEL can learn these patterns if we give it the flags.
# =============================================================================

def create_missing_flags(df, columns):
    """Create binary indicators for missing values."""
    out = pd.DataFrame(index=df.index)
    for col in columns:
        if col in df.columns:
            out[f'{col}_missing'] = df[col].isna().astype(int)
    return out


# Show the insight: missing values correlate with return_pct!
print('Average return by missingness status (training fold):')
print('(If these differ, missingness carries signal)\n')
for col in MISSING_FLAG_COLS[:5]:  # show first 5
    if col in train_fold.columns:
        present = train_fold.loc[train_fold[col].notna(), 'return_pct'].mean()
        missing = train_fold.loc[train_fold[col].isna(), 'return_pct'].mean()
        diff = missing - present
        marker = ' ← notable!' if abs(diff) > 5 else ''
        print(f'  {col:20s}  present={present:6.1f}%   missing={missing:6.1f}%   diff={diff:+.1f}%{marker}')

Average return by missingness status (training fold):
(If these differ, missingness carries signal)

  pe_ttm                present=  22.1%   missing=  14.3%   diff=-7.8% ← notable!
  price_to_book         present=  21.3%   missing=  21.6%   diff=+0.2%
  gross_margin          present=  26.2%   missing=  18.5%   diff=-7.7% ← notable!
  dividend_yield        present=  18.4%   missing=  22.6%   diff=+4.2%
  debt_to_equity        present=  20.4%   missing=  23.1%   diff=+2.7%


In [14]:
# =============================================================================
# STEP 7: Interaction Features
#
# WHAT:  Multiply two features together to capture their combined effect.
#
# WHY:   Sometimes what matters is the COMBINATION of two things, not each alone.
#
# Example: Revenue growth alone is ambiguous:
#   - A cheap stock with high growth = potentially great (undervalued grower)
#   - An expensive stock with high growth = might already be priced in
#
# By creating growth × valuation interactions, the model can distinguish
# these scenarios without needing to discover the pattern on its own.
# =============================================================================

def create_interactions(df):
    """Create interaction features between key financial dimensions."""
    out = pd.DataFrame(index=df.index)
    
    # Growth × Valuation interactions
    if 'revenue_growth_yoy' in df.columns and 'pe_ttm' in df.columns:
        # High growth + low P/E = potentially undervalued grower
        out['growth_x_pe'] = df['revenue_growth_yoy'] * df['pe_ttm']
    
    if 'revenue_growth_yoy' in df.columns and 'price_to_book' in df.columns:
        out['growth_x_pb'] = df['revenue_growth_yoy'] * df['price_to_book']
    
    # Profitability × Valuation
    if 'roe' in df.columns and 'price_to_book' in df.columns:
        # High ROE + low P/B = "quality at a reasonable price"
        out['roe_x_pb'] = df['roe'] * df['price_to_book']
    
    if 'net_margin' in df.columns and 'price_to_sales' in df.columns:
        out['margin_x_ps'] = df['net_margin'] * df['price_to_sales']
    
    # Size × Leverage
    if 'debt_to_equity' in df.columns and 'roa' in df.columns:
        # High debt + low returns = risky
        out['debt_x_roa'] = df['debt_to_equity'] * df['roa']
    
    return out


demo_interactions = create_interactions(train_fold.head(5))
print(f'Created {len(demo_interactions.columns)} interaction features:')
print(list(demo_interactions.columns))

Created 5 interaction features:
['growth_x_pe', 'growth_x_pb', 'roe_x_pb', 'margin_x_ps', 'debt_x_roa']


## 5. Putting It All Together: The Full Feature Pipeline

Now we combine all 7 steps into a single function that transforms any DataFrame
(train, validation, or test) into a feature matrix ready for modeling.

```
Raw Data  →  Clip Outliers  →  Log Transforms  →  Financial Ratios
          →  Margin Spreads  →  Sector Z-Scores →  Missing Flags
          →  Interactions    →  Final Feature Matrix
```

In [15]:
# =============================================================================
# THE COMPLETE FEATURE ENGINEERING FUNCTION
#
# This is the heart of our strategy. It takes raw data and produces
# a clean feature matrix.
#
# Parameters:
#   df           → the raw dataframe (train, valid, or test)
#   clip_bounds  → precomputed from training fold (Step 1)
#   sector_means → precomputed from training fold (Step 5)
#   sector_stds  → precomputed from training fold (Step 5)
#   etc.
#
# Returns:
#   A clean DataFrame with all engineered features, ready for modeling.
# =============================================================================

def build_feature_matrix(df, clip_bounds, sector_means, sector_stds,
                         global_means, global_stds):
    """
    Complete feature engineering pipeline.
    
    Steps:
    1. Start with raw data
    2. Clip extreme values using training-fold bounds
    3. Keep start_year as a feature
    4. Log-transform scale columns
    5. Add financial ratios
    6. Add margin spreads  
    7. Add sector z-scores
    8. Add missingness flags (computed BEFORE imputation!)
    9. Add interaction features
    10. Drop non-feature columns
    11. Keep only numeric columns
    """
    data = df.copy()
    
    # Step 1: Clip outliers
    data = apply_clip(data, clip_bounds)
    
    # Step 6 (early): Create missing flags BEFORE imputation
    # (if we do it after, all missing values are already filled and we lose the info)
    missing_flags = create_missing_flags(data, MISSING_FLAG_COLS)
    
    # Step 2: Log-transform scale columns
    for col in SCALE_COLS:
        if col in data.columns:
            data[f'log_{col}'] = signed_log1p(data[col])
    
    # Step 3: Create financial ratios (computed on clipped but not-yet-logged data)
    ratios = create_financial_ratios(data)
    
    # Step 4: Margin spreads
    spreads = create_margin_spreads(data)
    
    # Step 5: Sector z-scores
    available_zscore = [c for c in SECTOR_ZSCORE_COLS if c in data.columns]
    sector_z = apply_sector_zscores(
        data, available_zscore,
        sector_means, sector_stds,
        global_means, global_stds
    )
    
    # Step 7: Interactions (on clipped data)
    interactions = create_interactions(data)
    
    # Step 8: Drop non-feature columns
    drop_cols = [c for c in DROP_COLS if c in data.columns]
    data = data.drop(columns=drop_cols)
    
    # Combine everything
    data = pd.concat([data, ratios, spreads, sector_z, missing_flags, interactions], axis=1)
    
    # Step 9: Keep only numeric columns
    numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
    data = data[numeric_cols]
    
    return data


print('Feature pipeline defined. Let\'s apply it...')

Feature pipeline defined. Let's apply it...


In [16]:
# =============================================================================
# Apply the pipeline to all three datasets
# =============================================================================

X_train = build_feature_matrix(
    train_fold, clip_bounds,
    sector_means, sector_stds, global_means, global_stds
)
y_train = train_fold['return_pct'].values

X_valid = build_feature_matrix(
    valid_fold, clip_bounds,
    sector_means, sector_stds, global_means, global_stds
)
y_valid = valid_fold['return_pct'].values

X_test = build_feature_matrix(
    test, clip_bounds,
    sector_means, sector_stds, global_means, global_stds
)

print(f'Training features:   {X_train.shape[0]:,} rows × {X_train.shape[1]} features')
print(f'Validation features: {X_valid.shape[0]:,} rows × {X_valid.shape[1]} features')
print(f'Test features:       {X_test.shape[0]:,} rows × {X_test.shape[1]} features')
print()

# Show all feature names grouped
print('All features:')
for i, col in enumerate(X_train.columns, 1):
    print(f'  {i:3d}. {col}')

Training features:   16,436 rows × 84 features
Validation features: 6,634 rows × 84 features
Test features:       8,520 rows × 84 features

All features:
    1. start_year
    2. pe_ttm
    3. price_to_book
    4. price_to_sales
    5. growth_pe_ratio
    6. gross_margin
    7. operating_margin
    8. net_margin
    9. roa
   10. roe
   11. rote
   12. revenue_growth_3y
   13. revenue_growth_yoy
   14. revenue_ttm
   15. net_income_ttm
   16. income_before_tax
   17. eps_basic
   18. eps_diluted
   19. total_assets
   20. stockholders_equity
   21. current_assets
   22. current_liabilities
   23. long_term_debt
   24. goodwill
   25. inventory
   26. current_ratio
   27. quick_ratio
   28. debt_to_equity
   29. dividend_yield
   30. dividends_ttm
   31. dividends_paid_ttm
   32. shares_outstanding
   33. shares_diluted
   34. sector_code
   35. log_revenue_ttm
   36. log_net_income_ttm
   37. log_income_before_tax
   38. log_total_assets
   39. log_stockholders_equity
   40. log_curren

In [17]:
# =============================================================================
# Quick sanity check: are there any infinite or crazy values?
# =============================================================================

# Replace infinities with NaN (some divisions might create them)
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_valid = X_valid.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

print('Missing values per dataset after feature engineering:')
print(f'  Train:      {X_train.isna().sum().sum():,} total NaNs across all cells')
print(f'  Validation: {X_valid.isna().sum().sum():,} total NaNs')
print(f'  Test:       {X_test.isna().sum().sum():,} total NaNs')
print()
print('This is expected. The models we use can handle missing values.')

Missing values per dataset after feature engineering:
  Train:      364,256 total NaNs across all cells
  Validation: 148,142 total NaNs
  Test:       220,952 total NaNs

This is expected. The models we use can handle missing values.


## 6. Winsorize the Target (Training Only)

### Why clip the training target?

Some stocks have extreme returns (e.g., +10,571%). When the model tries to fit
these extreme values, it distorts predictions for normal stocks.

**Example:** Imagine the model adjusts its predictions to try to catch a 10,000%
outlier. This might cause it to predict +200% for a stock that actually goes to +20%.
The error on normal stocks goes up, and since RMSE squares errors, this is very costly.

**Solution:** Clip training targets to the 1st–99th percentile range. We tell the model:
"don't worry about returns more extreme than these bounds."

**Note:** We only clip the TRAINING targets. Validation targets stay as-is so we
measure real-world performance.

In [18]:
# =============================================================================
# Target winsorization
# =============================================================================

TARGET_CLIP_LOW = train_fold['return_pct'].quantile(0.01)
TARGET_CLIP_HIGH = train_fold['return_pct'].quantile(0.99)

print(f'Clipping training targets to [{TARGET_CLIP_LOW:.1f}%, {TARGET_CLIP_HIGH:.1f}%]')
print(f'  Before clipping: mean={y_train.mean():.1f}%, std={y_train.std():.1f}%')

y_train_clipped = np.clip(y_train, TARGET_CLIP_LOW, TARGET_CLIP_HIGH)

print(f'  After clipping:  mean={y_train_clipped.mean():.1f}%, std={y_train_clipped.std():.1f}%')
print()
print(f'Affected {(y_train != y_train_clipped).sum()} out of {len(y_train)} training rows.')

Clipping training targets to [-81.0%, 327.9%]
  Before clipping: mean=21.4%, std=159.0%
  After clipping:  mean=16.2%, std=64.3%

Affected 330 out of 16436 training rows.


## 7. Model Training

We train three different models and then combine them.

### Why these three models?

| Model | How it works | Strengths | Weaknesses |
|-------|-------------|-----------|------------|
| **Ridge** | Draws a line through the data (linear regression with regularization) | Stable, fast, captures broad trends | Can't capture non-linear patterns |
| **LightGBM** | Builds many small decision trees, each fixing the previous tree's mistakes | Handles missing values, captures complex patterns | Can overfit on small data |
| **XGBoost** | Similar to LightGBM but with a different algorithm | Same strengths, often gives different errors | Same weaknesses |

### Why combine them?

Each model makes different mistakes. By averaging their predictions, the mistakes
partly cancel out. This is called an **ensemble** and almost always works in Kaggle.

In [19]:
# =============================================================================
# MODEL 1: Ridge Regression (Linear Model)
#
# Ridge regression is a linear model — it finds the best straight-line
# relationship between each feature and the target.
#
# The 'alpha' parameter controls regularization: it prevents the model from
# becoming too confident about any single feature. Think of it as a
# "don't overreact" knob.
#
# Ridge REQUIRES:
#   - No missing values (we use SimpleImputer to fill them with the median)
#   - Scaled features (we use StandardScaler to normalize)
# =============================================================================

ridge_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # fill NaN with column median
    ('scaler', StandardScaler()),                   # normalize to mean=0, std=1
    ('model', Ridge(alpha=10.0)),                   # the actual model
])

# Fit on training data with clipped targets
ridge_pipeline.fit(X_train, y_train_clipped)

# Predict on validation
ridge_preds = ridge_pipeline.predict(X_valid)
ridge_rmse = rmse(y_valid, ridge_preds)

print(f'Ridge Regression RMSE on validation: {ridge_rmse:.4f}')

Ridge Regression RMSE on validation: 65.6090


In [20]:
# =============================================================================
# MODEL 2: LightGBM (Gradient Boosted Trees)
#
# LightGBM builds decision trees sequentially. Each new tree focuses on
# correcting the mistakes of the previous trees.
#
# Key parameters explained:
#   n_estimators    = how many trees to build (more = potentially better but slower)
#   learning_rate   = how much each tree contributes (smaller = more conservative)
#   max_depth       = how deep each tree can be (deeper = more complex patterns)
#   num_leaves      = max number of end-nodes per tree
#   min_child_samples = minimum rows in a leaf (higher = more conservative)
#   subsample       = fraction of rows used per tree (adds randomness, reduces overfitting)
#   colsample_bytree = fraction of features used per tree (same idea)
#   reg_alpha / reg_lambda = regularization (penalize complexity)
#
# LightGBM can handle missing values natively — no imputation needed!
# =============================================================================

lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    num_leaves=31,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1,  # suppress training output
    n_jobs=-1,
)

lgb_model.fit(
    X_train, y_train_clipped,
    eval_set=[(X_valid, y_valid)],
)

lgb_preds = lgb_model.predict(X_valid)
lgb_rmse = rmse(y_valid, lgb_preds)

print(f'LightGBM RMSE on validation: {lgb_rmse:.4f}')

LightGBM RMSE on validation: 67.0034


In [21]:
# =============================================================================
# MODEL 3: XGBoost (Gradient Boosted Trees — different algorithm)
#
# XGBoost is similar to LightGBM but uses a different tree-growing strategy.
# Having two tree-based models is useful because they make different mistakes,
# which helps when we average them later.
# =============================================================================

xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbosity=0,  # suppress output
    n_jobs=-1,
)

# XGBoost also handles missing values natively
xgb_model.fit(
    X_train, y_train_clipped,
    eval_set=[(X_valid, y_valid)],
    verbose=False,
)

xgb_preds = xgb_model.predict(X_valid)
xgb_rmse = rmse(y_valid, xgb_preds)

print(f'XGBoost RMSE on validation: {xgb_rmse:.4f}')

XGBoost RMSE on validation: 67.0789


In [22]:
# =============================================================================
# BASELINE: What RMSE do we get by just predicting the average?
#
# This is the simplest possible "model" — just predict the mean return
# for every stock. Any useful model should beat this.
# =============================================================================

mean_baseline_pred = np.full(len(y_valid), y_train.mean())
median_baseline_pred = np.full(len(y_valid), np.median(y_train))

mean_rmse = rmse(y_valid, mean_baseline_pred)
median_rmse = rmse(y_valid, median_baseline_pred)

print('\n' + '='*65)
print('  MODEL COMPARISON (Validation RMSE — lower is better)')
print('='*65)
print(f'  Mean baseline:     {mean_rmse:.4f}  ← just predicting the average')
print(f'  Median baseline:   {median_rmse:.4f}  ← just predicting the median')
print(f'  Ridge:             {ridge_rmse:.4f}')
print(f'  LightGBM:          {lgb_rmse:.4f}')
print(f'  XGBoost:           {xgb_rmse:.4f}')
print('='*65)


  MODEL COMPARISON (Validation RMSE — lower is better)
  Mean baseline:     65.3906  ← just predicting the average
  Median baseline:   65.3509  ← just predicting the median
  Ridge:             65.6090
  LightGBM:          67.0034
  XGBoost:           67.0789


## 8. Ensemble: Combining Models

### How does ensembling work?

We simply average the predictions of all three models:

```
ensemble_prediction = (ridge_pred + lgb_pred + xgb_pred) / 3
```

This works because:
- Ridge captures linear trends (e.g., "cheaper stocks generally do better")
- LightGBM captures non-linear patterns (e.g., "cheap stocks do better, but only if they are also profitable")
- XGBoost catches patterns LightGBM misses, and vice versa

When one model overestimates and another underestimates, the average is closer
to the truth. Statisticians call this **variance reduction**.

In [23]:
# =============================================================================
# Simple average ensemble
# =============================================================================

ensemble_preds = (ridge_preds + lgb_preds + xgb_preds) / 3
ensemble_rmse = rmse(y_valid, ensemble_preds)

print(f'Ensemble RMSE (average of all 3): {ensemble_rmse:.4f}')
print()

# Let's also try different blending weights
# Tree models usually perform better, so we might weight them higher
weighted_preds = 0.2 * ridge_preds + 0.4 * lgb_preds + 0.4 * xgb_preds
weighted_rmse = rmse(y_valid, weighted_preds)
print(f'Weighted ensemble (20% Ridge + 40% LGB + 40% XGB): {weighted_rmse:.4f}')

# Also try just tree models
tree_only_preds = 0.5 * lgb_preds + 0.5 * xgb_preds
tree_only_rmse = rmse(y_valid, tree_only_preds)
print(f'Tree-only ensemble (50% LGB + 50% XGB):            {tree_only_rmse:.4f}')

# Pick the best ensemble
ensemble_options = {
    'equal_avg': (ensemble_rmse, ensemble_preds, [1/3, 1/3, 1/3]),
    'weighted': (weighted_rmse, weighted_preds, [0.2, 0.4, 0.4]),
    'tree_only': (tree_only_rmse, tree_only_preds, [0.0, 0.5, 0.5]),
}
best_ensemble_name = min(ensemble_options, key=lambda k: ensemble_options[k][0])
best_rmse, best_preds, best_weights = ensemble_options[best_ensemble_name]

print(f'\nBest ensemble: {best_ensemble_name} with RMSE = {best_rmse:.4f}')
print(f'Weights: Ridge={best_weights[0]}, LGB={best_weights[1]}, XGB={best_weights[2]}')

Ensemble RMSE (average of all 3): 65.5330

Weighted ensemble (20% Ridge + 40% LGB + 40% XGB): 65.9949
Tree-only ensemble (50% LGB + 50% XGB):            66.9752

Best ensemble: equal_avg with RMSE = 65.5330
Weights: Ridge=0.3333333333333333, LGB=0.3333333333333333, XGB=0.3333333333333333


## 9. Feature Importance: What Actually Matters?

Feature importance tells us which columns the model relies on most.
This is useful for:
- Understanding the model's logic
- Deciding where to focus future feature engineering
- Spotting potential problems (e.g., if `start_year` is the top feature, the model
  might just be learning year-specific patterns that won't generalize)

In [24]:
# =============================================================================
# LightGBM feature importance
#
# 'gain' importance = how much each feature improves the model's predictions
# when it's used in a split. Higher = more important.
# =============================================================================

lgb_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': lgb_model.feature_importances_,
}).sort_values('importance', ascending=False)

print('Top 25 Most Important Features (LightGBM):')
print('='*55)
for i, (_, row) in enumerate(lgb_importance.head(25).iterrows(), 1):
    bar = '█' * int(row['importance'] / lgb_importance['importance'].max() * 30)
    print(f'  {i:2d}. {row["feature"]:35s} {row["importance"]:6.0f}  {bar}')

# Plot top 30
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

top30 = lgb_importance.head(30).sort_values('importance')
axes[0].barh(top30['feature'], top30['importance'], color='#2196F3')
axes[0].set_title('LightGBM Feature Importance (Top 30)', fontsize=13)
axes[0].set_xlabel('Importance (split gain)')

# XGBoost importance for comparison
xgb_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_,
}).sort_values('importance', ascending=False)

top30_xgb = xgb_importance.head(30).sort_values('importance')
axes[1].barh(top30_xgb['feature'], top30_xgb['importance'], color='#FF9800')
axes[1].set_title('XGBoost Feature Importance (Top 30)', fontsize=13)
axes[1].set_xlabel('Importance (split gain)')

plt.tight_layout()
plt.savefig(str(ROOT / 'notebooks' / 'feature_importance.png'), dpi=120, bbox_inches='tight')
plt.show()
print('\nSaved feature importance plot to notebooks/feature_importance.png')

Top 25 Most Important Features (LightGBM):
   1. start_year                             354  ██████████████████████████████
   2. price_to_sales                         352  █████████████████████████████
   3. earnings_yield                         306  █████████████████████████
   4. shares_diluted                         277  ███████████████████████
   5. price_to_book                          251  █████████████████████
   6. working_capital_ratio                  239  ████████████████████
   7. price_to_book_sector_z                 223  ██████████████████
   8. revenue_ttm                            222  ██████████████████
   9. pe_ttm                                 221  ██████████████████
  10. gross_minus_operating                  210  █████████████████
  11. operating_margin                       204  █████████████████
  12. eps_basic                              198  ████████████████
  13. price_to_sales_sector_z                197  ████████████████
  14. operating_minus_net 


Saved feature importance plot to notebooks/feature_importance.png


## 10. Error Analysis: Where Does the Model Struggle?

Looking at the biggest prediction errors helps us understand:
- Are certain sectors harder to predict?
- Are extreme returns (very high or very low) driving the RMSE?
- Are there patterns in what the model gets wrong?

In [25]:
# =============================================================================
# Analyze prediction errors
# =============================================================================

error_analysis = pd.DataFrame({
    'actual': y_valid,
    'predicted': best_preds,
    'error': y_valid - best_preds,
    'abs_error': np.abs(y_valid - best_preds),
    'sector_code': valid_fold['sector_code'].values,
})

print('Worst 15 predictions (highest absolute error):')
print('='*70)
worst = error_analysis.nlargest(15, 'abs_error')
for _, row in worst.iterrows():
    print(f'  Actual: {row["actual"]:+8.1f}%   Predicted: {row["predicted"]:+8.1f}%'
          f'   Error: {row["error"]:+8.1f}%   Sector: {row["sector_code"]}')

print(f'\nThese {len(worst)} stocks account for '
      f'{(worst["abs_error"]**2).sum() / (error_analysis["abs_error"]**2).sum() * 100:.1f}%'
      f' of total squared error.')

Worst 15 predictions (highest absolute error):
  Actual:  +1932.8%   Predicted:    +37.6%   Error:  +1895.2%   Sector: 3.0
  Actual:  +1667.1%   Predicted:    +49.5%   Error:  +1617.5%   Sector: 3.0
  Actual:  +1050.0%   Predicted:    -21.2%   Error:  +1071.2%   Sector: 1.0
  Actual:  +1016.9%   Predicted:    +86.5%   Error:   +930.4%   Sector: 4.0
  Actual:   +922.2%   Predicted:    +35.7%   Error:   +886.6%   Sector: 0.0
  Actual:   +792.5%   Predicted:    -31.1%   Error:   +823.5%   Sector: 1.0
  Actual:   +637.5%   Predicted:    +36.6%   Error:   +600.9%   Sector: 1.0
  Actual:   +588.0%   Predicted:     +5.2%   Error:   +582.8%   Sector: 3.0
  Actual:   +586.8%   Predicted:    +53.8%   Error:   +533.1%   Sector: 2.0
  Actual:   +517.7%   Predicted:     -4.0%   Error:   +521.7%   Sector: 0.0
  Actual:   +478.0%   Predicted:    -36.1%   Error:   +514.1%   Sector: 2.0
  Actual:   +471.4%   Predicted:    -27.2%   Error:   +498.6%   Sector: 0.0
  Actual:   +474.6%   Predicted:    -22.8

In [26]:
# =============================================================================
# RMSE by sector — which sectors are hardest to predict?
# =============================================================================

print('\nRMSE by sector:')
print('='*50)
sector_rmse = []
for sector in sorted(error_analysis['sector_code'].dropna().unique()):
    mask = error_analysis['sector_code'] == sector
    sector_data = error_analysis.loc[mask]
    s_rmse = rmse(sector_data['actual'], sector_data['predicted'])
    sector_rmse.append({'sector': int(sector), 'rmse': s_rmse, 'count': len(sector_data)})
    print(f'  Sector {int(sector):2d}:  RMSE = {s_rmse:7.2f}  ({len(sector_data)} stocks)')

sector_rmse_df = pd.DataFrame(sector_rmse)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(sector_rmse_df['sector'].astype(str), sector_rmse_df['rmse'], color='#4CAF50')
ax.set_title('Validation RMSE by Sector', fontsize=13)
ax.set_xlabel('Sector Code')
ax.set_ylabel('RMSE')
ax.axhline(y=best_rmse, color='red', linestyle='--', label=f'Overall RMSE: {best_rmse:.2f}')
ax.legend()
plt.tight_layout()
plt.savefig(str(ROOT / 'notebooks' / 'sector_rmse.png'), dpi=120, bbox_inches='tight')
plt.show()


RMSE by sector:
  Sector  0:  RMSE =   67.22  (1105 stocks)
  Sector  1:  RMSE =   63.05  (1099 stocks)
  Sector  2:  RMSE =   59.36  (1010 stocks)
  Sector  3:  RMSE =  106.58  (945 stocks)
  Sector  4:  RMSE =   57.66  (821 stocks)
  Sector  5:  RMSE =   30.80  (439 stocks)
  Sector  6:  RMSE =   39.92  (333 stocks)
  Sector  7:  RMSE =   34.50  (294 stocks)
  Sector  8:  RMSE =   33.38  (241 stocks)
  Sector  9:  RMSE =   31.53  (207 stocks)
  Sector 10:  RMSE =   35.05  (106 stocks)


## 11. Expanding Window Validation

Our single 2022 holdout is good but limited. To be more confident, we use an
**expanding window** — training on increasingly more data and validating on the next year:

```
Fold 1: Train on 2019        → Validate on 2020
Fold 2: Train on 2019–2020   → Validate on 2021
Fold 3: Train on 2019–2021   → Validate on 2022
```

If the model performs consistently across all folds, we can trust it more.
If it only works on one fold, it might be overfitting to that specific year.

In [27]:
# =============================================================================
# Expanding window cross-validation
# =============================================================================

print('Expanding Window Validation')
print('='*70)

cv_results = []

for valid_year in [2020, 2021, 2022]:
    # Split
    cv_train = train[train['start_year'] < valid_year]
    cv_valid = train[train['start_year'] == valid_year]
    
    if len(cv_train) == 0 or len(cv_valid) == 0:
        continue
    
    # Compute bounds from this fold's training data
    cv_clip_bounds = compute_clip_bounds(cv_train, all_numeric_cols)
    cv_avail = [c for c in SECTOR_ZSCORE_COLS if c in cv_train.columns]
    cv_s_means, cv_s_stds, cv_g_means, cv_g_stds = compute_sector_stats(
        cv_train, cv_avail
    )
    
    # Build features
    cv_X_train = build_feature_matrix(
        cv_train, cv_clip_bounds,
        cv_s_means, cv_s_stds, cv_g_means, cv_g_stds
    ).replace([np.inf, -np.inf], np.nan)
    
    cv_X_valid = build_feature_matrix(
        cv_valid, cv_clip_bounds,
        cv_s_means, cv_s_stds, cv_g_means, cv_g_stds
    ).replace([np.inf, -np.inf], np.nan)
    
    cv_y_train = cv_train['return_pct'].values
    cv_y_valid = cv_valid['return_pct'].values
    
    # Clip training target
    cv_tgt_lo = np.percentile(cv_y_train, 1)
    cv_tgt_hi = np.percentile(cv_y_train, 99)
    cv_y_train_c = np.clip(cv_y_train, cv_tgt_lo, cv_tgt_hi)
    
    # Train LightGBM (fastest to train)
    cv_lgb = lgb.LGBMRegressor(
        n_estimators=500, learning_rate=0.03, max_depth=5,
        num_leaves=31, min_child_samples=50,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, verbose=-1, n_jobs=-1,
    )
    cv_lgb.fit(cv_X_train, cv_y_train_c)
    cv_lgb_preds = cv_lgb.predict(cv_X_valid)
    cv_lgb_rmse = rmse(cv_y_valid, cv_lgb_preds)
    
    # Also compute median baseline for comparison
    cv_baseline_rmse = rmse(cv_y_valid, np.full(len(cv_y_valid), np.median(cv_y_train)))
    
    improvement = (cv_baseline_rmse - cv_lgb_rmse) / cv_baseline_rmse * 100
    
    cv_results.append({
        'valid_year': valid_year,
        'train_rows': len(cv_train),
        'valid_rows': len(cv_valid),
        'baseline_rmse': cv_baseline_rmse,
        'lgb_rmse': cv_lgb_rmse,
        'improvement_%': improvement,
    })
    
    print(f'  Fold: Train <{valid_year}, Validate {valid_year}')
    print(f'    Train: {len(cv_train):,} rows, Valid: {len(cv_valid):,} rows')
    print(f'    Baseline RMSE: {cv_baseline_rmse:.2f}, LightGBM RMSE: {cv_lgb_rmse:.2f}'
          f'  ({improvement:+.1f}% vs baseline)')
    print()

cv_df = pd.DataFrame(cv_results)
print('\nSummary:')
display(cv_df.round(2))
print(f'\nAverage LightGBM RMSE across folds: {cv_df["lgb_rmse"].mean():.2f}')

Expanding Window Validation


  Fold: Train <2020, Validate 2020
    Train: 5,029 rows, Valid: 5,339 rows
    Baseline RMSE: 266.90, LightGBM RMSE: 262.29  (+1.7% vs baseline)



  Fold: Train <2021, Validate 2021
    Train: 10,368 rows, Valid: 6,068 rows
    Baseline RMSE: 49.94, LightGBM RMSE: 78.69  (-57.6% vs baseline)



  Fold: Train <2022, Validate 2022
    Train: 16,436 rows, Valid: 6,634 rows
    Baseline RMSE: 65.35, LightGBM RMSE: 67.00  (-2.5% vs baseline)


Summary:


,valid_year,train_rows,valid_rows,baseline_rmse,lgb_rmse,improvement_%
0,2020,5029,5339,266.9000,262.2900,1.7300
1,2021,10368,6068,49.9400,78.6900,-57.5600
2,2022,16436,6634,65.3500,67.0000,-2.5300



Average LightGBM RMSE across folds: 135.99


## 12. Generate Final Submission

Now we:
1. Retrain all models on the **full** training data (all years 2019–2022)
2. Predict on the test set
3. Create the submission file in the expected format

In [28]:
# =============================================================================
# Retrain on all training data and generate submission
# =============================================================================

# Build features for the full training set (all years)
# Use the full training set for computing bounds and sector stats
full_clip_bounds = compute_clip_bounds(train, all_numeric_cols)
full_avail = [c for c in SECTOR_ZSCORE_COLS if c in train.columns]
full_s_means, full_s_stds, full_g_means, full_g_stds = compute_sector_stats(
    train, full_avail
)

X_full = build_feature_matrix(
    train, full_clip_bounds,
    full_s_means, full_s_stds, full_g_means, full_g_stds
).replace([np.inf, -np.inf], np.nan)

X_test_final = build_feature_matrix(
    test, full_clip_bounds,
    full_s_means, full_s_stds, full_g_means, full_g_stds
).replace([np.inf, -np.inf], np.nan)

y_full = train['return_pct'].values

# Clip full training target
full_tgt_lo = np.percentile(y_full, 1)
full_tgt_hi = np.percentile(y_full, 99)
y_full_clipped = np.clip(y_full, full_tgt_lo, full_tgt_hi)

print(f'Full training set: {X_full.shape[0]:,} rows × {X_full.shape[1]} features')
print(f'Test set:          {X_test_final.shape[0]:,} rows × {X_test_final.shape[1]} features')
print(f'Target clipped to [{full_tgt_lo:.1f}%, {full_tgt_hi:.1f}%]')

Full training set: 23,070 rows × 84 features
Test set:          8,520 rows × 84 features
Target clipped to [-80.2%, 299.2%]


In [29]:
# =============================================================================
# Retrain all three models on full data
# =============================================================================

# Ridge
final_ridge = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=10.0)),
])
final_ridge.fit(X_full, y_full_clipped)
test_ridge_preds = final_ridge.predict(X_test_final)
print('Ridge: trained on full data ✓')

# LightGBM
final_lgb = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.03, max_depth=5,
    num_leaves=31, min_child_samples=50,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, verbose=-1, n_jobs=-1,
)
final_lgb.fit(X_full, y_full_clipped)
test_lgb_preds = final_lgb.predict(X_test_final)
print('LightGBM: trained on full data ✓')

# XGBoost
final_xgb = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.03, max_depth=5,
    min_child_weight=50, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, verbosity=0, n_jobs=-1,
)
final_xgb.fit(X_full, y_full_clipped)
test_xgb_preds = final_xgb.predict(X_test_final)
print('XGBoost: trained on full data ✓')

Ridge: trained on full data ✓


LightGBM: trained on full data ✓


XGBoost: trained on full data ✓


In [30]:
# =============================================================================
# Create ensemble predictions and save submission
# =============================================================================

# Use the best ensemble weights we found during validation
w_ridge, w_lgb, w_xgb = best_weights
test_ensemble_preds = (
    w_ridge * test_ridge_preds +
    w_lgb * test_lgb_preds +
    w_xgb * test_xgb_preds
)

# Create submission DataFrame
submission = sample_submission.copy()
submission['return_pct'] = test_ensemble_preds

# Verify format
assert list(submission.columns) == ['id', 'return_pct'], 'Wrong columns!'
assert len(submission) == len(sample_submission), 'Wrong row count!'
assert submission['return_pct'].isna().sum() == 0, 'NaN predictions!'

# Save
submission_path = SUBMISSION_DIR / 'feature_engineering_ensemble.csv'
submission.to_csv(submission_path, index=False)

print(f'Submission saved to: {submission_path}')
print(f'Rows: {len(submission):,}')
print(f'Prediction range: [{submission["return_pct"].min():.1f}%, {submission["return_pct"].max():.1f}%]')
print(f'Mean prediction: {submission["return_pct"].mean():.1f}%')
print(f'\nEnsemble weights used: Ridge={w_ridge}, LGB={w_lgb}, XGB={w_xgb}')
print()
display(submission.head(10))

Submission saved to: C:\Users\joni0\kaggle-competition-stock-return-fundamentals\submissions\feature_engineering_ensemble.csv
Rows: 8,520
Prediction range: [-63.4%, 156.7%]
Mean prediction: 6.3%

Ensemble weights used: Ridge=0.3333333333333333, LGB=0.3333333333333333, XGB=0.3333333333333333



,id,return_pct
0,0,-8.9239
1,1,10.5491
2,2,2.4346
3,3,-0.1537
4,4,5.4127
5,5,11.8765
6,6,1.3335
7,7,1.5229
8,8,21.7461
9,9,3.6171


## 13. Summary and Next Steps

### What we built:

1. **Feature pipeline** with 7 types of engineered features:
   - Winsorized raw features (clip outliers)
   - Log-transformed scale features (revenue, assets, etc.)
   - Financial ratios (earnings yield, asset turnover, leverage, etc.)
   - Margin spreads (gross-operating, operating-net, ROE-ROA)
   - Sector-relative z-scores (how does each stock compare to sector peers?)
   - Missingness flags (is a missing value informative?)
   - Interaction features (growth × valuation, quality × price, etc.)

2. **Three models** with proper validation:
   - Ridge (linear baseline)
   - LightGBM (tree-based, handles non-linear patterns)
   - XGBoost (second tree model for diversity)

3. **Ensemble** that combines all three models

4. **Feature importance** to understand what drives predictions

5. **Error analysis** to understand where the model struggles

---

### What each member can work on next:

| Task | Difficulty | Expected Impact |
|------|-----------|----------------|
| Tune LightGBM hyperparameters (try different max_depth, learning_rate) | Easy | Medium |
| Add more interaction features (test different combinations) | Easy | Low–Medium |
| Try CatBoost as a 4th model | Easy | Medium |
| Experiment with different target clipping bounds (e.g., 2%–98%) | Easy | Medium |
| Create time-based features (e.g., quarter-over-quarter changes if data allows) | Medium | Medium |
| Try sector-specific models (train separate models per sector) | Medium | Medium |
| Add polynomial features for top-important features | Medium | Low–Medium |
| Optimize ensemble weights with cross-validation | Medium | Medium |
| Investigate PCA or feature selection to reduce noise | Medium | Low |
| Try target transformation (e.g., predicting log returns) | Hard | High |
| Build a stacking ensemble (use model predictions as features for a meta-model) | Hard | High |

---

### Golden rules:

1. **Always validate on a time split** — never random split
2. **Compute everything from training data only** — no peeking at validation/test
3. **Small RMSE improvements matter** — don't chase flashy but unstable tricks
4. **Document your experiments** — so others can build on your work
5. **When in doubt, keep it simple** — a clean Ridge model beats a buggy neural network